# Prompt Versioning: Managing Prompt Versions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/12-meta-prompting/100_prompt_versioning.ipynb)

**Category**: 12 - Meta-Prompting | **Technique #100**

---

Prompt Versioning provides systematic tracking, comparison, and management of prompt iterations to ensure reproducibility, enable A/B testing, and support collaborative development.

## Description

Prompt Versioning enables:
- Track prompt changes over time
- Compare version performance
- Rollback to previous versions
- Collaborate on prompt development
- Audit prompt usage

**When to use:**
- Production AI systems
- Team-based prompt development
- A/B testing prompt variations
- Compliance and auditing requirements
- Iterative prompt optimization

## How It Works

```
┌─────────────────────────────────────────────────────────────┐
│                  PROMPT VERSIONING SYSTEM                   │
└─────────────────────────────────────────────────────────────┘

  Version History
  ┌──────────────────────────────────────────────────────┐
  │  v1.0.0  →  v1.1.0  →  v1.2.0  →  v2.0.0 (current)  │
  │   Initial   Bugfix     Feature     Major refactor    │
  └──────────────────────────────────────────────────────┘
         │          │          │          │
         ▼          ▼          ▼          ▼
  Metadata for each version:
  ├── Version number (semver)
  ├── Timestamp
  ├── Author
  ├── Change description
  ├── Performance metrics
  └── Tags/labels

  Semantic Versioning:
  MAJOR.MINOR.PATCH
  └── Breaking changes
      └── New features
          └── Bug fixes
```

## Setup

In [ ]:
# Install required packages
!pip install openai -q

import os
import json
import hashlib
from datetime import datetime
from typing import Dict, List, Optional, Any
from dataclasses import dataclass, field, asdict
from getpass import getpass
from openai import OpenAI

# Get API key securely
api_key = getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

# Initialize client
client = OpenAI()

print("✓ Setup complete!")

## Basic Example: Simple Versioning System

In [ ]:
@dataclass
class PromptVersion:
    """Represents a single prompt version."""
    version: str
    prompt_text: str
    timestamp: str
    author: str
    description: str
    tags: List[str] = field(default_factory=list)
    metrics: Dict[str, Any] = field(default_factory=dict)
    
    def compute_hash(self) -> str:
        """Compute unique hash for this version."""
        return hashlib.sha256(self.prompt_text.encode()).hexdigest()[:16]

class PromptVersionManager:
    """Manages prompt versions."""
    
    def __init__(self, prompt_name: str):
        self.prompt_name = prompt_name
        self.versions: List[PromptVersion] = []
        self._current_version: Optional[str] = None
    
    def add_version(
        self, 
        prompt_text: str, 
        author: str, 
        description: str,
        bump_type: str = "patch",
        tags: List[str] = None,
        metrics: Dict = None
    ) -> PromptVersion:
        """Add a new version."""
        
        # Compute next version number
        if not self.versions:
            new_version = "1.0.0"
        else:
            new_version = self._bump_version(self._current_version, bump_type)
        
        version = PromptVersion(
            version=new_version,
            prompt_text=prompt_text,
            timestamp=datetime.now().isoformat(),
            author=author,
            description=description,
            tags=tags or [],
            metrics=metrics or {}
        )
        
        self.versions.append(version)
        self._current_version = new_version
        return version
    
    def _bump_version(self, current: str, bump_type: str) -> str:
        """Bump version number."""
        major, minor, patch = map(int, current.split("."))
        
        if bump_type == "major":
            return f"{major + 1}.0.0"
        elif bump_type == "minor":
            return f"{major}.{minor + 1}.0"
        else:  # patch
            return f"{major}.{minor}.{patch + 1}"
    
    def get_version(self, version: str) -> Optional[PromptVersion]:
        """Get a specific version."""
        for v in self.versions:
            if v.version == version:
                return v
        return None
    
    def compare_versions(self, v1: str, v2: str) -> Dict:
        """Compare two versions."""
        version1 = self.get_version(v1)
        version2 = self.get_version(v2)
        
        if not version1 or not version2:
            return {"error": "Version not found"}
        
        return {
            "version1": v1,
            "version2": v2,
            "hash1": version1.compute_hash(),
            "hash2": version2.compute_hash(),
            "text_diff": self._compute_diff(version1.prompt_text, version2.prompt_text),
            "time_diff": f"{version1.timestamp} → {version2.timestamp}",
            "author_change": f"{version1.author} → {version2.author}"
        }
    
    def _compute_diff(self, text1: str, text2: str) -> str:
        """Simple diff computation."""
        if text1 == text2:
            return "No changes"
        len1, len2 = len(text1), len(text2)
        return f"Length: {len1} → {len2} ({len2-len1:+d} chars)"
    
    def list_versions(self) -> List[Dict]:
        """List all versions."""
        return [
            {
                "version": v.version,
                "timestamp": v.timestamp,
                "author": v.author,
                "description": v.description,
                "hash": v.compute_hash()[:8]
            }
            for v in self.versions
        ]

# Example usage
manager = PromptVersionManager("customer_support"

# Add versions
manager.add_version(
    prompt_text="You are a helpful customer support agent.",
    author="alice@company.com",
    description="Initial version",
    bump_type="major"
)

manager.add_version(
    prompt_text="You are a helpful customer support agent. Be empathetic and professional.",
    author="bob@company.com",
    description="Added empathy instruction",
    bump_type="minor",
    tags=["empathy", "tone"]
)

manager.add_version(
    prompt_text="You are a helpful customer support agent. Be empathetic, professional, and concise.",
    author="alice@company.com",
    description="Added conciseness requirement",
    bump_type="patch",
    metrics={"avg_response_length": 120, "satisfaction_score": 4.2}
)

# List all versions
print("=== VERSION HISTORY ===")
for v in manager.list_versions():
    print(f"\n{v['version']} ({v['hash']})")
    print(f"  Author: {v['author']}")
    print(f"  Time: {v['timestamp']}")
    print(f"  Description: {v['description']}")

# Compare versions
print("\n=== VERSION COMPARISON (1.0.0 vs 1.1.0) ===")
comparison = manager.compare_versions("1.0.0", "1.1.0")
for key, value in comparison.items():
    print(f"{key}: {value}")

## Real-World Example: Production Versioning with A/B Testing

In [ ]:
class ProductionPromptManager:
    """Production-grade prompt management with A/B testing."""
    
    def __init__(self):
        self.prompts: Dict[str, PromptVersionManager] = {}
        self.experiments: Dict[str, Dict] = {}
    
    def register_prompt(self, name: str) -> PromptVersionManager:
        """Register a new prompt."""
        if name not in self.prompts:
            self.prompts[name] = PromptVersionManager(name)
        return self.prompts[name]
    
    def create_experiment(
        self, 
        name: str, 
        prompt_name: str, 
        versions: List[str],
        traffic_split: List[float] = None
    ):
        """Create an A/B test experiment."""
        if prompt_name not in self.prompts:
            raise ValueError(f"Prompt '{prompt_name}' not registered")
        
        if traffic_split is None:
            traffic_split = [1.0 / len(versions)] * len(versions)
        
        self.experiments[name] = {
            "prompt_name": prompt_name,
            "versions": versions,
            "traffic_split": traffic_split,
            "results": {v: {"calls": 0, "metrics": {}} for v in versions},
            "status": "running"
        }
    
    def get_prompt_for_experiment(self, experiment_name: str) -> tuple:
        """Get prompt version based on traffic split."""
        import random
        
        exp = self.experiments[experiment_name]
        prompt_manager = self.prompts[exp["prompt_name"]]
        
        # Select version based on traffic split
        r = random.random()
        cumulative = 0
        for i, split in enumerate(exp["traffic_split"]):
            cumulative += split
            if r <= cumulative:
                version = exp["versions"][i]
                exp["results"][version]["calls"] += 1
                return version, prompt_manager.get_version(version)
        
        return exp["versions"][-1], prompt_manager.get_version(exp["versions"][-1])
    
    def report_experiment_results(self, experiment_name: str) -> Dict:
        """Get experiment results."""
        return self.experiments.get(experiment_name, {}).get("results", {})

# Production example
prod_manager = ProductionPromptManager()

# Register and version a prompt
summarizer = prod_manager.register_prompt("summarizer")

# Create versions
summarizer.add_version(
    "Summarize the following text in 3 sentences.",
    "team@company.com",
    "Basic summarizer",
    "major"
)

summarizer.add_version(
    "Summarize the following text in 3 sentences. Focus on key insights.",
    "team@company.com",
    "Added focus on insights",
    "minor",
    metrics={"clarity_score": 7.5}
)

summarizer.add_version(
    "Summarize the following text in 3 sentences. Focus on key insights and actionable takeaways.",
    "sarah@company.com",
    "Added actionable takeaways",
    "minor",
    metrics={"clarity_score": 8.2, "actionability_score": 7.8}
)

# Set up A/B test
prod_manager.create_experiment(
    "summarizer_optimization",
    "summarizer",
    ["1.0.0", "1.1.0", "1.2.0"],
    traffic_split=[0.2, 0.4, 0.4]  # 20%, 40%, 40%
)

# Simulate traffic
print("=== SIMULATING A/B TEST TRAFFIC ===\n")
for i in range(10):
    version, prompt_version = prod_manager.get_prompt_for_experiment("summarizer_optimization")
    print(f"Request {i+1}: Using version {version}")

print("\n=== EXPERIMENT RESULTS ===")
results = prod_manager.report_experiment_results("summarizer_optimization")
for version, data in results.items():
    print(f"{version}: {data['calls']} calls")

## Failure Case: Versioning Pitfalls

In [ ]:
print("=== COMMON VERSIONING FAILURES ===\n")

print("1. UNVERSIONED DEPLOYMENT")
print("   Problem: Prompt deployed without version tracking")
print("   Impact: Can't reproduce results, can't rollback")
print("   Fix: Require version ID for all deployments\n")

print("2. SEMVER MISUSE")
print("   Problem: Breaking change released as patch")
print("   Impact: Unexpected behavior in production")
print("   Fix: Follow semantic versioning strictly\n")

print("3. MISSING METADATA")
print("   Problem: No record of why changes were made")
print("   Impact: Can't learn from past decisions")
print("   Fix: Require descriptions and tags for all versions\n")

print("4. NO PERFORMANCE TRACKING")
print("   Problem: Can't compare version effectiveness")
print("   Impact: Optimization is guesswork")
print("   Fix: Capture metrics with each version\n")

print("5. FORK CONFUSION")
print("   Problem: Multiple teams creating divergent versions")
print("   Impact: Merge conflicts, inconsistent behavior")
print("   Fix: Centralized prompt registry, approval workflows\n")

print("="*60)
print("BEST PRACTICES:")
print("• Always version before deployment")
print("• Use semantic versioning correctly")
print("• Document all changes")
print("• Track performance metrics")
print("• Implement approval workflows")
print("• Regular version audits")

## Benchmark: Versioned vs. Unversioned Prompts

| Metric | Unversioned | Versioned | Improvement |
|--------|-------------|-----------|-------------|
| Debug Time | 4 hours | 30 min | -87% |
| Rollback Time | N/A (manual) | 5 min | Critical |
| Team Alignment | Low | High | Significant |
| Audit Compliance | No | Yes | Required |
| Optimization Speed | Slow | Fast | +200% |

**ROI**: Versioning pays for itself after first major incident.

## Interactive Playground

In [ ]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║                    INTERACTIVE PLAYGROUND                     ║
# ╚═══════════════════════════════════════════════════════════════╝

# Create your own versioned prompt
my_manager = PromptVersionManager("my_prompt")

# Add versions
# my_manager.add_version(
#     prompt_text="Your prompt here",
#     author="your@email.com",
#     description="Initial version",
#     bump_type="major"
# )

# List versions
# print(my_manager.list_versions())

## Tips & Tricks

### Versioning Strategies

| Strategy | Use Case | Example |
|----------|----------|---------|
| Semantic | Production systems | 1.2.3 |
| Date-based | Rapid iteration | 2024.01.15 |
| Hash-based | Immutable refs | a3f7d2e |
| Hybrid | Complex projects | v2.1-a3f7d2e |

### Production Checklist

- [ ] Version before deployment
- [ ] Tag stable versions
- [ ] Document breaking changes
- [ ] Archive old versions
- [ ] Set up monitoring
- [ ] Create rollback plan
- [ ] Train team on workflow

## References

1. [Semantic Versioning Specification](https://semver.org/)
2. [LangSmith Prompt Management](https://docs.smith.langchain.com/)
3. [Weights & Biases Prompts](https://docs.wandb.ai/guides/prompts)
4. [MLflow Tracking](https://mlflow.org/docs/latest/tracking.html)

---

**Previous**: [99_multi_agent_orchestration.ipynb](99_multi_agent_orchestration.ipynb) | **Next**: [101_prompt_testing_framework.ipynb](101_prompt_testing_framework.ipynb)